# Mobile Price Classification — Data Understanding (Tahap 1)
        **Praktikum 14.16 | CRISP-DM | Dataset Kaggle**

        **Tujuan:** Memahami struktur data, kelengkapan, duplikasi, dan distribusi target.

        **Batasan:** belum training 3 algoritma, belum preprocessing, dan belum tuning. Tahap-tahap tersebut dikerjakan kemudian, secara berurutan.

## Cara menjalankan di Google Colab
        1. Buka [Google Colab](https://colab.research.google.com/), pilih **File → Upload notebook** dan unggah file `.ipynb` ini.
        2. Jalankan sel satu per satu dari atas ke bawah (**Shift + Enter**).
        3. Saat diminta, unggah **`train.csv`** dari dataset Kaggle *Mobile Price Classification*; `test.csv` belum diperlukan.
        4. Grafik tampil di bawah sel dan otomatis tersimpan ke folder `figures/`.

        Jika `train.csv` sudah ada di folder notebook, dialog upload tidak akan muncul.
**Catatan:** Di Colab, grafik tampil langsung di bawah sel dan dapat dilihat dengan scroll. Tombol **Previous / Next** tersedia di file `.py` yang dijalankan dari terminal Windows.


## Memuat dataset `train.csv`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Dapat dijalankan di Google Colab maupun Jupyter lokal.
# Jika train.csv belum ditemukan, Google Colab akan meminta upload.
calon_file = [
    Path('Mobile_Price_Classification/train.csv'),
    Path('train.csv'),
    Path('/content/train.csv'),
]
file_csv = next((p for p in calon_file if p.is_file()), None)

if file_csv is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            'train.csv belum ditemukan. Taruh di folder notebook atau '
            'di folder Mobile_Price_Classification/'
        ) from exc
    print('Silakan upload train.csv dari dataset Kaggle Mobile Price Classification.')
    diupload = files.upload()
    if 'train.csv' not in diupload:
        raise FileNotFoundError('File yang diupload harus bernama train.csv.')
    file_csv = Path('train.csv')

df = pd.read_csv(file_csv)
if 'price_range' not in df.columns:
    raise ValueError('Kolom target price_range tidak ditemukan. Gunakan train.csv, bukan test.csv.')

print('Dataset:', file_csv)
print('Ukuran:', df.shape)

## 1. Melihat 5 baris pertama dan dimensi dataset

In [ ]:
print('=== 5 BARIS PERTAMA ===')
display(df.head())
print(f'Jumlah baris: {df.shape[0]:,}')
print(f'Jumlah kolom: {df.shape[1]} (termasuk 1 target)')
print(f'Jumlah fitur: {df.shape[1] - 1}')

## 2. Tipe data dan informasi kolom

In [ ]:
print('=== INFO DATASET ===')
df.info()
print('\n=== TIPE DATA ===')
display(df.dtypes.value_counts().rename_axis('Tipe data').reset_index(name='Jumlah kolom'))

## 3. Memeriksa missing value dan duplikat

In [ ]:
missing_per_kolom = df.isna().sum().sort_values(ascending=False)
jumlah_duplikat = int(df.duplicated().sum())
print('=== MISSING VALUE ===')
display(missing_per_kolom.rename('Jumlah missing').to_frame())
print('Total missing value:', int(missing_per_kolom.sum()))
print('Total baris duplikat:', jumlah_duplikat)

## 4. Menampilkan jumlah anggota setiap kelas

In [ ]:
label_kelas = {0: '0 - Low', 1: '1 - Medium', 2: '2 - High', 3: '3 - Very High'}
jumlah_kelas = df['price_range'].value_counts().sort_index()
proporsi_kelas = (jumlah_kelas / len(df) * 100).round(2)
tabel_kelas = pd.DataFrame({
    'Kategori': [label_kelas.get(int(k), str(k)) for k in jumlah_kelas.index],
    'Jumlah data': jumlah_kelas.values,
    'Proporsi (%)': proporsi_kelas.values,
})
display(tabel_kelas)

## 5. Visualisasi: distribusi kelas target

In [ ]:
OUTPUT = Path('figures/01_data_understanding')
OUTPUT.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 4.8))
bar = ax.bar([label_kelas.get(int(x), str(x)) for x in jumlah_kelas.index], jumlah_kelas.values)
ax.bar_label(bar, padding=4, fontsize=10)
ax.set_title('Distribusi Kelas Harga Smartphone')
ax.set_xlabel('Kategori price_range')
ax.set_ylabel('Jumlah smartphone')
ax.set_ylim(0, max(jumlah_kelas.values) * 1.2)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(OUTPUT / '01_distribusi_kelas.png', dpi=160, bbox_inches='tight')
plt.show()

## 6. Visualisasi: pemeriksaan kelengkapan data

In [ ]:
jumlah_isi = df.notna().sum().sort_values()
kelengkapan = jumlah_isi / len(df) * 100
fig, ax = plt.subplots(figsize=(9.5, 6.5))
ax.barh(kelengkapan.index, kelengkapan.values)
ax.set_xlim(0, 108)
ax.set_xlabel('Persentase nilai terisi (%)')
ax.set_title('Kelengkapan Setiap Kolom Dataset')
ax.grid(axis='x', alpha=0.2)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(OUTPUT / '02_kelengkapan_kolom.png', dpi=160, bbox_inches='tight')
plt.show()

## 7. Ringkasan otomatis untuk catatan laporan ilmiah

In [ ]:
jumlah_target = df['price_range'].nunique()
imbang = jumlah_kelas.nunique() == 1
print('RINGKASAN DATA UNDERSTANDING')
print(f'- Dataset memiliki {df.shape[0]} baris, {df.shape[1]-1} fitur, dan 1 target (price_range).')
print(f'- Terdapat {jumlah_target} kategori harga.')
print(f'- Missing value: {int(missing_per_kolom.sum())}.')
print(f'- Baris duplikat: {jumlah_duplikat}.')
print(f'- Distribusi kelas {"seimbang" if imbang else "tidak sepenuhnya seimbang"}.')
print('- Hasil ini merupakan pemeriksaan awal; belum dilakukan preprocessing atau pemodelan.')
print(f'- Grafik tersimpan di: {OUTPUT}')

### Catatan untuk laporan ilmiah
        Simpan hasil tabel dan grafik. Pada laporan, **jelaskan angka yang benar-benar muncul saat notebook dijalankan**. Jika memakai data lain atau preprocessing berbeda, hasil juga dapat berubah.

        Jangan menyimpulkan bahwa korelasi membuktikan sebab-akibat atau menyebut data dihapus jika baru dideteksi sebagai outlier.

## Opsional — Unduh semua grafik dari Google Colab

In [ ]:
from pathlib import Path
from zipfile import ZipFile
folder_grafik = Path('figures')
file_zip = Path('grafik_mobile_price_tahap_' + '1' + '.zip')
# Kode ini membuat arsip semua grafik yang ada; tahap 1/2 tidak saling tergantung.
with ZipFile(file_zip, 'w') as arsip:
    for gambar in folder_grafik.rglob('*.png'):
        arsip.write(gambar, gambar.as_posix())
print('Arsip grafik:', file_zip)
try:
    from google.colab import files
    files.download(str(file_zip))
except ImportError:
    print('Anda menjalankan notebook di luar Colab, unduh berkas zip secara manual.')